# Companion Notebook 07: Decoding Strategies Verification

This notebook verifies the mathematical execution of **Temperature Scaling**, **Repetition Penalties**, and **Top-k / Top-p (Nucleus) sampling** from scratch.


## 1. Step-by-Step Hand-Calculation Verification
We run the exact scenario defined in the study guide:
- Vocabulary size $|V| = 3$.
- Raw Logits: $\mathbf{z} = [2.0, 1.0, -1.0]$
- Temperature: $T = 0.5$
- Top-k: $k = 2$
- Repetition Penalty: $\theta = 1.2$ applied to index 0 (which has already been generated).


In [1]:
import numpy as np
import torch

# Define inputs
logits = np.array([2.0, 1.0, -1.0])
temp = 0.5
k = 2
rep_penalty = 1.2
generated_tokens = {0} # index 0 has already been generated

print("Raw Input Logits:", logits)

# Step 1: Apply Repetition Penalty
adjusted_logits = logits.copy()
for idx in range(len(logits)):
    if idx in generated_tokens:
        val = adjusted_logits[idx]
        if val > 0:
            adjusted_logits[idx] = val / rep_penalty
        else:
            adjusted_logits[idx] = val * rep_penalty

print("Step 1: Logits after Repetition Penalty:", adjusted_logits)

# Step 2: Apply Temperature Scaling
scaled_logits = adjusted_logits / temp
print("Step 2: Logits after Temperature Scaling:", scaled_logits)

# Step 3: Apply Top-k Sampling (k=2)
# Sort indices in descending order
sorted_indices = np.argsort(scaled_logits)[::-1]
pruned_logits = scaled_logits.copy()
# Identify indices to prune (any index not in top k)
prune_indices = sorted_indices[k:]
pruned_logits[prune_indices] = -np.inf
print("Step 3: Logits after Top-k pruning:", pruned_logits)

# Step 4: Evaluate Softmax
# Compute exponentials (avoiding exp(-inf) warnings)
exps = np.exp(pruned_logits)
exps[np.isneginf(pruned_logits)] = 0.0
probs = exps / np.sum(exps)
print("Step 4: Softmax Probabilities:", probs)

# Verification check
expected_probs = np.array([0.7914, 0.2086, 0.0000])
print("\nMatches study guide calculations (within precision)?")
print(np.allclose(probs, expected_probs, atol=1e-3))


Raw Input Logits: [ 2.  1. -1.]
Step 1: Logits after Repetition Penalty: [ 1.66666667  1.         -1.        ]
Step 2: Logits after Temperature Scaling: [ 3.33333333  2.         -2.        ]
Step 3: Logits after Top-k pruning: [3.33333333 2.               -inf]
Step 4: Softmax Probabilities: [0.79139147 0.20860853 0.        ]

Matches study guide calculations (within precision)?
True


## 2. Top-p (Nucleus) Sampling Implementation
We implement the Top-p gating function in PyTorch and trace its behavior on a sample probability distribution.


In [2]:
def top_p_logits(logits, p):
    # Sort logits descending
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    # Cumulative sum of softmax probabilities
    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    
    # Identify tokens to remove (cumulative probability exceeding threshold p)
    # We keep the first token that exceeds the threshold (hence we shift right)
    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    
    # Map back to original indices
    indices_to_remove = sorted_indices[sorted_indices_to_remove]
    # Set logits to -inf
    logits_pruned = logits.clone()
    logits_pruned[indices_to_remove] = float('-inf')
    return logits_pruned

# Example logits for vocabulary size = 5
logits_example = torch.tensor([2.5, 2.0, 1.8, 0.5, 0.1])
p_threshold = 0.90

pruned_res = top_p_logits(logits_example, p_threshold)
probs_res = torch.softmax(pruned_res, dim=-1)

print("Original Logits:     ", logits_example.numpy())
print("Original Softmax:    ", torch.softmax(logits_example, dim=-1).numpy())
print("Pruned Logits (p=0.9):", pruned_res.numpy())
print("Pruned Softmax:      ", probs_res.numpy())


Original Logits:      [2.5 2.  1.8 0.5 0.1]
Original Softmax:     [0.42933762 0.26040643 0.21320274 0.05810453 0.03894863]
Pruned Logits (p=0.9): [ 2.5  2.   1.8 -inf -inf]
Pruned Softmax:       [0.47548494 0.2883962  0.23611882 0.         0.        ]
